# 24.1 Genetic algorithms

A population searches by selection, recombination, mutation, and preservation of the best discoveries.

Genetic algorithms answer how we search when objectives are discontinuous, noisy, or built from choices rather than smooth weights. Fitness scores selection tickets, crossover recombines useful structure, mutation restores exploration, and elitism preserves rare good solutions.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 241
rng = np.random.default_rng(SEED)


def shifted_selection_probabilities(fitness):
    fitness = np.asarray(fitness, dtype=float)
    shifted = fitness - np.min(fitness) + 1.0
    probabilities = shifted / np.sum(shifted)
    return probabilities


def one_dimensional_fitness(x):
    x = np.asarray(x, dtype=float)
    value = -np.square(x - 3.0) + 10.0
    return value


def sphere_fitness(points):
    target = np.array([1.0, -1.0])
    loss = np.sum(np.square(points - target), axis=1)
    return -loss


def rastrigin_fitness(points):
    points = np.asarray(points, dtype=float)
    dim = points.shape[1]
    loss = 10.0 * dim + np.sum(np.square(points) - 10.0 * np.cos(2.0 * np.pi * points), axis=1)
    return -loss


def constrained_fitness(points):
    target = np.array([1.5, -1.5])
    loss = np.sum(np.square(points - target), axis=1)
    violation = np.maximum(0.0, points[:, 0] + points[:, 1] - 0.5)
    penalty = 25.0 * np.square(violation)
    return -(loss + penalty)


def make_ga_ladder():
    ladder = []
    ladder.append({"name": "D1 1-D quadratic fitness", "dim": 1, "bounds": (-1.0, 7.0), "objective": lambda x: one_dimensional_fitness(x[:, 0])})
    ladder.append({"name": "D2 2-D sphere target", "dim": 2, "bounds": (-5.0, 5.0), "objective": sphere_fitness})
    ladder.append({"name": "D3 2-D Rastrigin", "dim": 2, "bounds": (-5.12, 5.12), "objective": rastrigin_fitness})
    ladder.append({"name": "D4 constrained box plus penalty", "dim": 2, "bounds": (-4.0, 4.0), "objective": constrained_fitness})
    ladder.append({"name": "D5 20-D Rastrigin", "dim": 20, "bounds": (-5.12, 5.12), "objective": rastrigin_fitness})
    return ladder


def blend_crossover(parents, rng):
    weights = rng.uniform(0.25, 0.75, size=(parents.shape[0] // 2, 1))
    left = parents[0::2]
    right = parents[1::2]
    children_a = weights * left + (1.0 - weights) * right
    children_b = weights * right + (1.0 - weights) * left
    children = np.vstack([children_a, children_b])
    return children


def genetic_optimize(objective, dim, bounds, population_size=40, generations=45, mutation_scale=0.25, elite_count=2, pressure=1.0, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    lower, upper = bounds
    population = rng.uniform(lower, upper, size=(population_size, dim))
    history = []
    snapshots = []
    for generation in range(generations):
        fitness = objective(population)
        order = np.argsort(fitness)[::-1]
        elites = population[order[:elite_count]].copy()
        weights = fitness - np.min(fitness) + 1.0
        weights = np.power(weights, pressure)
        probabilities = weights / np.sum(weights)
        parent_indices = rng.choice(population_size, size=population_size - elite_count, replace=True, p=probabilities)
        parents = population[parent_indices]
        if parents.shape[0] % 2 == 1:
            parents = np.vstack([parents, parents[-1]])
        children = blend_crossover(parents, rng)
        children = children[:population_size - elite_count]
        mutation = rng.normal(0.0, mutation_scale, size=children.shape)
        children = np.clip(children + mutation, lower, upper)
        population = np.vstack([elites, children])
        history.append(float(np.max(fitness)))
        if generation in {0, generations // 2, generations - 1}:
            snapshots.append(population.copy())
    fitness = objective(population)
    best_index = int(np.argmax(fitness))
    result = {"best": population[best_index], "best_fitness": float(fitness[best_index]), "history": np.array(history), "population": population, "snapshots": snapshots}
    return result


## The concept, built once (D1): selection pressure without silencing exploration

The lesson's shifted fitness-proportionate rule is

$$p_i=\frac{F(x_i)-\min_j F(x_j)+1}{\sum_k (F(x_k)-\min_j F(x_j)+1)}.$$

For $F(x)=-(x-3)^2+10$ on $\{0,2,4,6\}$, the shifted weights are $\{1,9,9,1\}$ and the probabilities must be $\{0.05,0.45,0.45,0.05\}$.

In [ ]:

candidates = np.array([0.0, 2.0, 4.0, 6.0])
fitness = one_dimensional_fitness(candidates)
probabilities = shifted_selection_probabilities(fitness)

print("fitness:", fitness)
print("selection probabilities:", probabilities)

assert np.allclose(fitness, np.array([1.0, 9.0, 9.0, 1.0]))
assert np.allclose(probabilities, np.array([0.05, 0.45, 0.45, 0.05]))


## Add crossover, mutation, and elitism

Averaging parents $2$ and $4$ gives $3.0$. Adding mutation $+0.5$ gives child $3.5$, and the lesson fitness is $F(3.5)=9.75$. The reusable optimizer below applies that same selection-crossover-mutation-elitism loop to every rung.

In [ ]:

parent_a = 2.0
parent_b = 4.0
child = (parent_a + parent_b) / 2.0 + 0.5
child_fitness = float(one_dimensional_fitness(child))

demo_result = genetic_optimize(make_ga_ladder()[0]["objective"], dim=1, bounds=(-1.0, 7.0), population_size=20, generations=18, mutation_scale=0.20, rng=np.random.default_rng(SEED))

print("child:", child)
print("child fitness:", child_fitness)
print("demo best fitness:", demo_result["best_fitness"])

assert np.isclose(child, 3.5)
assert np.isclose(child_fitness, 9.75)
assert len(demo_result["history"]) == 18


## The dataset ladder

Family F4 has no shared ladder here. We build the D1–D5 objective ladder inline: known 1-D optimum, 2-D sphere, multimodal Rastrigin, constrained penalty objective, and 20-D Rastrigin.

In [ ]:

ladder = make_ga_ladder()

for rung in ladder:
    preview_rng = np.random.default_rng(SEED + rung["dim"])
    lower, upper = rung["bounds"]
    sample = preview_rng.uniform(lower, upper, size=(3, rung["dim"]))
    values = rung["objective"](sample)
    print(rung["name"], "shape", sample.shape, "sample fitness", np.round(values, 3))


In [ ]:

results = []

for index, rung in enumerate(ladder):
    result = genetic_optimize(rung["objective"], dim=rung["dim"], bounds=rung["bounds"], population_size=50, generations=50, mutation_scale=0.18, elite_count=2, pressure=0.9, rng=np.random.default_rng(SEED + index))
    results.append(result)
    print(f"{rung['name']}: best_fitness={result['best_fitness']:.4f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for index, rung in enumerate(ladder):
    ax = axes[0, index]
    lower, upper = rung["bounds"]
    if rung["dim"] == 1:
        grid = np.linspace(lower, upper, 200)
        values = rung["objective"](grid.reshape(-1, 1))
        ax.plot(grid, values)
        ax.scatter(results[index]["population"][:, 0], rung["objective"](results[index]["population"]), s=12)
    else:
        grid_x = np.linspace(lower, upper, 80)
        grid_y = np.linspace(lower, upper, 80)
        mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)
        points = np.column_stack([mesh_x.ravel(), mesh_y.ravel()])
        if rung["dim"] > 2:
            padded = np.zeros((points.shape[0], rung["dim"]))
            padded[:, :2] = points
            values = rung["objective"](padded)
            pop = results[index]["population"][:, :2]
        else:
            values = rung["objective"](points)
            pop = results[index]["population"]
        ax.contourf(mesh_x, mesh_y, values.reshape(mesh_x.shape), levels=20, cmap="viridis")
        ax.scatter(pop[:, 0], pop[:, 1], c="white", s=10, edgecolor="black")
    ax.set_title(rung["name"].split()[0])

for index, result in enumerate(results):
    axes[1, index].plot(result["history"])
    axes[1, index].set_title("best fitness")
    axes[1, index].set_xlabel("generation")

fig.suptitle("Population-on-landscape panels and best-fitness curves")
plt.tight_layout()


## Pitfall on D5: selection pressure too high

Winner-heavy selection can collapse variance before the 20-D Rastrigin population has enough time to explore. The fix is softer selection, elitism, and nonzero mutation.

In [ ]:

d5 = ladder[-1]

bad = genetic_optimize(d5["objective"], dim=d5["dim"], bounds=d5["bounds"], population_size=50, generations=50, mutation_scale=0.03, elite_count=1, pressure=6.0, rng=np.random.default_rng(999))
good = genetic_optimize(d5["objective"], dim=d5["dim"], bounds=d5["bounds"], population_size=50, generations=50, mutation_scale=0.20, elite_count=2, pressure=0.9, rng=np.random.default_rng(999))

bad_variance = float(np.mean(np.var(bad["population"], axis=0)))
good_variance = float(np.mean(np.var(good["population"], axis=0)))

print("bad best fitness", bad["best_fitness"], "variance", bad_variance)
print("fixed best fitness", good["best_fitness"], "variance", good_variance)


## Evaluate it + Practice

- Compare the reported best fitness against a seeded random-search baseline with the same evaluation budget.
- Sanity check: D1 must hit the lesson's worked calculation before trusting D2–D5.
- Ablation: increase selection pressure and shrink mutation should make the hardest rung worse or less stable.
- Failure signals: population variance collapses too early, bounds are violated, or the summary curve improves only by one lucky sample.
- Re-run with another seed only after the structural checks pass; do not tune from a single trace.

Practice prompts:
1. Change the hardest rung budget and explain whether convergence improves per objective call.
2. Add one diagnostic for diversity and plot it next to the metric.
3. Swap the D3 multimodal objective and predict which operator needs retuning.